# Multi-Model Root Cause Analysis for Smart Manufacturing

This notebook implements an end-to-end multi-label Root Cause Analysis (RCA) pipeline for three smart manufacturing subsystems: **Coolant, Hydraulics, and Probe**.

The workflow follows:

**Raw fault telemetry → Feature extraction → Multi-label target creation → Preprocessing → MLP and 1D CNN → Late fusion → Top-k RCA → Explainable readable summary**

The notebook is structured for reproducibility and focuses on the final implementation rather than exploratory or debugging steps.


## 1. Dataset Acquisition and Environment Setup

This cell imports the required libraries and downloads the causRCA dataset. It establishes the raw data source required to reproduce the complete pipeline.

In [ ]:
import pandas as pd
import numpy as np
import json
import glob
import os

!pip install zenodo_get --quiet

!zenodo_get 10.5281/zenodo.15876410 -o causrca_data

## 2. Dataset Extraction

The downloaded archives are extracted into a consistent local directory structure so that the subsystem fault datasets can be accessed programmatically.

In [ ]:
import zipfile
import glob
import os

zip_files = glob.glob("causrca_data/*.zip")

for z in zip_files:
    with zipfile.ZipFile(z, "r") as f:
            f.extractall("causrca_data")

print("Extraction completed.")

## 3. Fault Dataset Discovery

This cell discovers the Coolant, Hydraulics, and Probe fault files. These file lists are used by the subsystem-specific feature extraction pipelines.

In [ ]:
coolant_files = glob.glob(
      "causrca_data/dig_twin/exp_coolant/**/faultDataset_*.csv",
          recursive=True
          )

hydraulics_files = glob.glob(
      "causrca_data/dig_twin/exp_hydraulics/**/faultDataset_*.csv",
         recursive=True
                  )

probe_files = glob.glob(
       "causrca_data/dig_twin/exp_probe/**/faultDataset_*.csv",
          recursive=True
                          )

print("Coolant:", len(coolant_files))
print("Hydraulics:", len(hydraulics_files))
print("Probe:", len(probe_files))


# 2. Coolant Root Cause Analysis

## 2.1 Feature Extraction and Ground-Truth Label Creation

For each Coolant fault case, this cell extracts fixed-length features from telemetry around the fault window. It captures continuous, binary, counter, categorical, and alarm behaviour before and after the root-cause start time. Ground-truth root causes are then encoded using `MultiLabelBinarizer` for multi-label classification.

In [ ]:
import pandas as pd
import numpy as np
import glob
import json
import os
from sklearn.preprocessing import MultiLabelBinarizer


# ============================================================
# STEP 1: FEATURE EXTRACTION FUNCTION
# ============================================================

def extract_coolant_features(file, window=30):

    # --------------------------------------------------------
    # Read CSV
    # --------------------------------------------------------

    df = pd.read_csv(file)

    # --------------------------------------------------------
    # Detect the time column automatically
    # --------------------------------------------------------

    possible_time_columns = [
        "time",
        "Time",
        "timestamp",
        "Timestamp",
        "time_s",
        "Time_s",
        "timestamp_s",
        "Timestamp_s",
        "relative_time",
        "RelativeTime"
    ]

    time_column = None

    for col in possible_time_columns:
        if col in df.columns:
            time_column = col
            break

    # If standard names are not found, search for a column
    # containing the word "time" or "timestamp"

    if time_column is None:

        for col in df.columns:

            col_lower = str(col).lower()

            if "time" in col_lower or "timestamp" in col_lower:
                time_column = col
                break

    if time_column is None:

        print("\nERROR: Could not find a time column.")
        print("File:", file)
        print("Available columns:")
        print(df.columns.tolist())

        raise KeyError(
            "No time/timestamp column found in the CSV."
        )

    # Convert time column to numeric
    df["time_numeric"] = pd.to_numeric(
        df[time_column],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Detect experiment and run
    # --------------------------------------------------------

    run_id = os.path.basename(
        os.path.dirname(file)
    )

    exp_id = os.path.basename(
        os.path.dirname(
            os.path.dirname(file)
        )
    )

    # Example:
    # exp_27/run_7
    case_name = f"{exp_id}/{run_id}"

    # --------------------------------------------------------
    # Read causes.json
    # --------------------------------------------------------

    causes_file = os.path.join(
        os.path.dirname(file),
        "causes.json"
    )

    with open(causes_file, "r") as f:
        causes = json.load(f)

    cause_start = causes["cause_start_at"]

    # --------------------------------------------------------
    # Convert value to numeric
    # --------------------------------------------------------

    df["value_numeric"] = pd.to_numeric(
        df["value"],
        errors="coerce"
    )

    # ========================================================
    # BEFORE WINDOW
    # ========================================================

    before = df[
        (df["time_numeric"] >= cause_start - window) &
        (df["time_numeric"] < cause_start)
    ].copy()

    # ========================================================
    # AFTER WINDOW
    # ========================================================

    after = df[
        (df["time_numeric"] >= cause_start) &
        (df["time_numeric"] <= cause_start + window)
    ].copy()

    # ========================================================
    # CONTINUOUS FEATURES
    # ========================================================

    continuous_nodes = df[
        df["type"].astype(str).str.lower() == "continuous"
    ]["node"].unique()

    continuous_features = {}

    for node in continuous_nodes:

        before_values = before.loc[
            before["node"] == node,
            "value_numeric"
        ].dropna()

        after_values = after.loc[
            after["node"] == node,
            "value_numeric"
        ].dropna()

        before_mean = (
            before_values.mean()
            if len(before_values) > 0
            else np.nan
        )

        after_mean = (
            after_values.mean()
            if len(after_values) > 0
            else np.nan
        )

        before_std = (
            before_values.std()
            if len(before_values) > 1
            else np.nan
        )

        after_std = (
            after_values.std()
            if len(after_values) > 1
            else np.nan
        )

        mean_change = (
            after_mean - before_mean
            if pd.notna(before_mean)
            and pd.notna(after_mean)
            else np.nan
        )

        relative_change = (
            mean_change / abs(before_mean)
            if pd.notna(mean_change)
            and pd.notna(before_mean)
            and before_mean != 0
            else np.nan
        )

        std_change = (
            after_std - before_std
            if pd.notna(before_std)
            and pd.notna(after_std)
            else np.nan
        )

        continuous_features[
            f"{node}_before_mean"
        ] = before_mean

        continuous_features[
            f"{node}_after_mean"
        ] = after_mean

        continuous_features[
            f"{node}_before_std"
        ] = before_std

        continuous_features[
            f"{node}_after_std"
        ] = after_std

        continuous_features[
            f"{node}_mean_change"
        ] = mean_change

        continuous_features[
            f"{node}_relative_change"
        ] = relative_change

        continuous_features[
            f"{node}_std_change"
        ] = std_change

    # ========================================================
    # BINARY FEATURES
    # ========================================================

    binary_nodes = df[
        df["type"].astype(str).str.lower() == "binary"
    ]["node"].unique()

    binary_features = {}

    for node in binary_nodes:

        before_values = before.loc[
            before["node"] == node,
            "value_numeric"
        ].dropna()

        after_values = after.loc[
            after["node"] == node,
            "value_numeric"
        ].dropna()

        before_ratio = (
            (before_values == 1).mean()
            if len(before_values) > 0
            else np.nan
        )

        after_ratio = (
            (after_values == 1).mean()
            if len(after_values) > 0
            else np.nan
        )

        before_transitions = (
            (before_values.diff().abs() > 0).sum()
            if len(before_values) > 1
            else np.nan
        )

        after_transitions = (
            (after_values.diff().abs() > 0).sum()
            if len(after_values) > 1
            else np.nan
        )

        binary_features[
            f"{node}_before_state_ratio"
        ] = before_ratio

        binary_features[
            f"{node}_after_state_ratio"
        ] = after_ratio

        binary_features[
            f"{node}_state_ratio_change"
        ] = (
            after_ratio - before_ratio
            if pd.notna(before_ratio)
            and pd.notna(after_ratio)
            else np.nan
        )

        binary_features[
            f"{node}_before_transitions"
        ] = before_transitions

        binary_features[
            f"{node}_after_transitions"
        ] = after_transitions

        binary_features[
            f"{node}_transition_change"
        ] = (
            after_transitions - before_transitions
            if pd.notna(before_transitions)
            and pd.notna(after_transitions)
            else np.nan
        )

    # ========================================================
    # COUNTER FEATURES
    # ========================================================

    counter_nodes = df[
        df["type"].astype(str).str.lower() == "counter"
    ]["node"].unique()

    counter_features = {}

    for node in counter_nodes:

        before_values = before.loc[
            before["node"] == node,
            "value_numeric"
        ].dropna()

        after_values = after.loc[
            after["node"] == node,
            "value_numeric"
        ].dropna()

        before_mean = (
            before_values.mean()
            if len(before_values) > 0
            else np.nan
        )

        after_mean = (
            after_values.mean()
            if len(after_values) > 0
            else np.nan
        )

        mean_change = (
            after_mean - before_mean
            if pd.notna(before_mean)
            and pd.notna(after_mean)
            else np.nan
        )

        before_rate = (
            (before_values.iloc[-1] - before_values.iloc[0])
            / (len(before_values) - 1)
            if len(before_values) > 1
            else np.nan
        )

        after_rate = (
            (after_values.iloc[-1] - after_values.iloc[0])
            / (len(after_values) - 1)
            if len(after_values) > 1
            else np.nan
        )

        rate_difference = (
            after_rate - before_rate
            if pd.notna(before_rate)
            and pd.notna(after_rate)
            else np.nan
        )

        counter_features[
            f"{node}_before_mean"
        ] = before_mean

        counter_features[
            f"{node}_after_mean"
        ] = after_mean

        counter_features[
            f"{node}_mean_change"
        ] = mean_change

        counter_features[
            f"{node}_before_rate"
        ] = before_rate

        counter_features[
            f"{node}_after_rate"
        ] = after_rate

        counter_features[
            f"{node}_rate_difference"
        ] = rate_difference

    # ========================================================
    # CATEGORICAL FEATURES
    # ========================================================

    categorical_nodes = df[
        df["type"].astype(str).str.lower() == "categorical"
    ]["node"].unique()

    categorical_features = {}

    for node in categorical_nodes:

        before_values = before[
            before["node"] == node
        ]["value"].dropna()

        after_values = after[
            after["node"] == node
        ]["value"].dropna()

        if len(before_values) > 0:

            before_props = (
                before_values
                .value_counts(normalize=True)
            )

            for value, proportion in before_props.items():

                categorical_features[
                    f"{node}_before_{value}"
                ] = proportion

        if len(after_values) > 0:

            after_props = (
                after_values
                .value_counts(normalize=True)
            )

            for value, proportion in after_props.items():

                categorical_features[
                    f"{node}_after_{value}"
                ] = proportion

    # ========================================================
    # ALARM FEATURES
    # ========================================================

    alarm_nodes = df[
        df["type"].astype(str).str.lower() == "alarm"
    ]["node"].unique()

    alarm_features = {}

    for node in alarm_nodes:

        before_values = before[
            before["node"] == node
        ]["value"].astype(str).str.lower()

        after_values = after[
            after["node"] == node
        ]["value"].astype(str).str.lower()

        before_active = (
            before_values.isin(
                ["true", "1", "active"]
            ).mean()
            if len(before_values) > 0
            else np.nan
        )

        after_active = (
            after_values.isin(
                ["true", "1", "active"]
            ).mean()
            if len(after_values) > 0
            else np.nan
        )

        alarm_features[
            f"{node}_before_ratio"
        ] = before_active

        alarm_features[
            f"{node}_after_ratio"
        ] = after_active

        alarm_features[
            f"{node}_ratio_change"
        ] = (
            after_active - before_active
            if pd.notna(before_active)
            and pd.notna(after_active)
            else np.nan
        )

    # ========================================================
    # COMBINE FEATURES
    # ========================================================

    features = {}

    features.update(continuous_features)
    features.update(binary_features)
    features.update(counter_features)
    features.update(categorical_features)
    features.update(alarm_features)

    # Metadata
    features["case_name"] = case_name
    features["cause_start_at"] = cause_start

    return pd.DataFrame([features])


# ============================================================
# STEP 2: FIND ALL COOLANT FAULT CSV FILES
# ============================================================

coolant_files = glob.glob(
    "causrca_data/dig_twin/exp_coolant/*/run_*/faultDataset_*.csv"
)

print("Number of coolant files found:", len(coolant_files))


# ============================================================
# STEP 3: EXTRACT FEATURES FROM ALL RUNS
# ============================================================

coolant_features_list = []

for file in coolant_files:

    features = extract_coolant_features(
        file,
        window=30
    )

    coolant_features_list.append(features)

coolant_features = pd.concat(
    coolant_features_list,
    ignore_index=True
)


# ============================================================
# STEP 5: DEFINE GROUND-TRUTH CAUSES
# ============================================================

experiment_causes = {

    "exp_27": [
        "CLT_Level_lt_Min",
        "HP_Pump_Ok"
    ],

    "exp_35": [
        "CLF_Filter_Ok",
        "LP_Pump_Ok"
    ],

    "exp_28": [
        "CLF_Filter_Ok",
        "F_Filter_Ok",
        "LP_Pump_Ok"
    ],

    "exp_22": [
        "HP_Pump_Ok",
        "LT_Level_Ok"
    ]
}


# ============================================================
# STEP 6: CREATE LABEL LIST
# ============================================================

labels = []

for case in coolant_features["case_name"]:

    # Example:
    # exp_27/run_7
    exp_id = str(case).split("/")[0]

    if exp_id in experiment_causes:

        labels.append(
            experiment_causes[exp_id]
        )

    else:

        labels.append([])


# ============================================================
# STEP 7: MULTI-LABEL ENCODING
# ============================================================

mlb_coolant = MultiLabelBinarizer()

y_coolant = mlb_coolant.fit_transform(labels)

y_coolant = pd.DataFrame(
    y_coolant,
    columns=mlb_coolant.classes_,
    index=coolant_features.index
)




## 2.2 Coolant Preprocessing

The feature matrix is prepared for deep learning by removing features with more than 80% missing values, performing a case-based train/test split, imputing remaining missing values using training-set medians, and standardizing features using `StandardScaler`. Fitting preprocessing components only on training data helps avoid data leakage.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


# ============================================================
# STEP 1: Separate metadata from ML features
# ============================================================

metadata_cols = [
    "case_name",
    "cause_start_at"
]

feature_cols = [
    col for col in coolant_features.columns
    if col not in metadata_cols
]

X_coolant = coolant_features[feature_cols].copy()

print("Original feature matrix:")
print(X_coolant.shape)


# ============================================================
# STEP 2: Remove extremely sparse features
# ============================================================

missing_percentage = (
    X_coolant.isna().mean() * 100
)

missing_threshold = 80

keep_features = missing_percentage[
    missing_percentage <= missing_threshold
].index

X_coolant_reduced = X_coolant[
    keep_features
].copy()

print("\nAfter removing features with >80% missing values:")
print(X_coolant_reduced.shape)

print(
    "Features removed:",
    X_coolant.shape[1] - X_coolant_reduced.shape[1]
)

print(
    "Remaining missing values:",
    X_coolant_reduced.isna().sum().sum()
)


# ============================================================
# STEP 3: Make sure X and y have the same rows
# ============================================================

assert len(X_coolant_reduced) == len(y_coolant), \
    "X and y have different numbers of rows!"

print("\nX and y row check: PASSED")


# ============================================================
# STEP 4: Split into training and testing data
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_coolant_reduced,
    y_coolant,
    test_size=0.20,
    random_state=42
)

print("\nTrain/Test split:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


# ============================================================
# STEP 5: Median imputation
# Fit ONLY on training data
# ============================================================

imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True
)

X_train_imputed = imputer.fit_transform(X_train)

X_test_imputed = imputer.transform(X_test)


# Convert back to DataFrames
X_train_imputed = pd.DataFrame(
    X_train_imputed,
    columns=X_train.columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    X_test_imputed,
    columns=X_test.columns,
    index=X_test.index
)


print("\nAfter imputation:")
print("Training missing values:",
      X_train_imputed.isna().sum().sum())

print("Testing missing values:",
      X_test_imputed.isna().sum().sum())


# ============================================================
# STEP 6: Standardization
# Fit ONLY on training data
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_imputed
)

X_test_scaled = scaler.transform(
    X_test_imputed
)


# Convert back to DataFrames
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=X_test.columns,
    index=X_test.index
)


# ============================================================
# STEP 7: Final verification
# ============================================================

print("\n" + "=" * 70)
print("FINAL PREPROCESSED DATA")
print("=" * 70)

print("\nX_train:", X_train_scaled.shape)
print("X_test :", X_test_scaled.shape)

print("\ny_train:", y_train.shape)
print("y_test :", y_test.shape)

print(
    "\nTraining NaN:",
    X_train_scaled.isna().sum().sum()
)

print(
    "Testing NaN:",
    X_test_scaled.isna().sum().sum()
)
print(
      "\nTraining mean (approximately 0):",
          round(X_train_scaled.mean().mean(), 6)
          )


print(
      "Training standard deviation (approximately 1):",
          round(X_train_scaled.std().mean(), 6)
          )

# Store final processed coolant test data in globally accessible variables
X_test_coolant_final = X_test_scaled
y_test_coolant_final = y_test

## 2.3 Coolant MLP Model

A Multi-Layer Perceptron (MLP) learns global nonlinear relationships across the engineered feature vector. Dense layers transform the input representation, while dropout and early stopping help reduce overfitting on the small multi-label dataset.

In [ ]:
# ============================================================
# COOLANT RCA - MLP MODEL
# ============================================================

import tensorflow as tf
import numpy as np
import pandas as pd

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# STEP 1: Convert data to NumPy arrays
# ============================================================

X_train_mlp = X_train_scaled.values
X_test_mlp = X_test_scaled.values

y_train_mlp = y_train.values
y_test_mlp = y_test.values


print("MLP input shape:")
print(X_train_mlp.shape)

print("\nMLP target shape:")
print(y_train_mlp.shape)


# ============================================================
# STEP 2: Build MLP architecture
# ============================================================

mlp_coolant = Sequential([
    Dense(
        64,
        activation="relu",
        input_shape=(X_train_mlp.shape[1],)
    ),
    Dropout(0.30),
    Dense(
        32,
        activation="relu"
    ),
    Dropout(0.20),
    Dense(
        6,
        activation="sigmoid"
    )
])


# ============================================================
# STEP 3: Compile model
# ============================================================

mlp_coolant.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="binary_accuracy"
        )
    ]
)


# ============================================================
# STEP 4: Display architecture
# ============================================================

mlp_coolant.summary()


# ============================================================
# STEP 5: Early stopping
# ============================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)


# ============================================================
# STEP 6: Train MLP
# ============================================================

history_mlp = mlp_coolant.fit(
    X_train_mlp,
    y_train_mlp,
    epochs=100,
    batch_size=4,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1
)


# ============================================================
# STEP 7: Evaluate on TEST SET
# ============================================================

test_loss, test_accuracy = mlp_coolant.evaluate(
    X_test_mlp,
    y_test_mlp,
    verbose=0
)


print("\n" + "=" * 70)
print("MLP TEST RESULTS")
print("=" * 70)

print("Test Loss:", test_loss)
print("Test Binary Accuracy:", test_accuracy)


# ============================================================
# STEP 8: Generate RCA probabilities
# ============================================================

y_pred_mlp_probability = mlp_coolant.predict(
    X_test_mlp,
    verbose=0
)


# ============================================================
# STEP 9: Convert probabilities to 0/1 predictions
# ============================================================

threshold = 0.5

y_pred_mlp = (
    y_pred_mlp_probability >= threshold
).astype(int)


# ============================================================
# STEP 10: Display predictions
# ============================================================

print("\nRoot-cause probabilities:")

probability_df = pd.DataFrame(
    y_pred_mlp_probability,
    columns=y_coolant.columns,
    index=y_test.index
)

display(probability_df)


print("\nMLP predicted root causes:")

prediction_df = pd.DataFrame(
    y_pred_mlp,
    columns=y_coolant.columns,
    index=y_test.index
)

display(prediction_df)


# ============================================================
# STEP 11: Compare ACTUAL vs PREDICTED
# ============================================================

print("\n" + "=" * 70)
print("ACTUAL vs PREDICTED ROOT CAUSES")
print("=" * 70)

comparison = pd.DataFrame({
    "case_name":
        coolant_features.loc[
            y_test.index,
            "case_name"
        ],
    "actual":
        y_test.apply(
            lambda row: list(
                row.index[row == 1]
            ),
            axis=1
        ),
    "predicted":
        prediction_df.apply(
            lambda row: list(
                row.index[row == 1]
            ),
            axis=1
        )
})

display(comparison)

## 2.4 Coolant MLP Evaluation

This cell evaluates the MLP using multi-label metrics including Hamming loss, precision, recall, F1-score, exact-match accuracy, and root-cause-wise performance.

In [ ]:
# ============================================================
# COOLANT RCA - MLP EVALUATION
# ============================================================

from sklearn.metrics import (
    hamming_loss,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)
import pandas as pd
import numpy as np


# ============================================================
# STEP 1: Calculate overall multi-label metrics
# ============================================================

mlp_hamming_loss = hamming_loss(
    y_test_mlp,
    y_pred_mlp
)

mlp_precision = precision_score(
    y_test_mlp,
    y_pred_mlp,
    average="micro",
    zero_division=0
)

mlp_recall = recall_score(
    y_test_mlp,
    y_pred_mlp,
    average="micro",
    zero_division=0
)

mlp_f1 = f1_score(
    y_test_mlp,
    y_pred_mlp,
    average="micro",
    zero_division=0
)

mlp_macro_f1 = f1_score(
    y_test_mlp,
    y_pred_mlp,
    average="macro",
    zero_division=0
)

mlp_exact_match = accuracy_score(
    y_test_mlp,
    y_pred_mlp
)


# ============================================================
# STEP 2: Display overall results
# ============================================================

print("=" * 70)
print("MLP - MULTI-LABEL RCA EVALUATION")
print("=" * 70)

print(f"\nHamming Loss       : {mlp_hamming_loss:.4f}")
print(f"Micro Precision   : {mlp_precision:.4f}")
print(f"Micro Recall      : {mlp_recall:.4f}")
print(f"Micro F1-score    : {mlp_f1:.4f}")
print(f"Macro F1-score    : {mlp_macro_f1:.4f}")
print(f"Exact Match Ratio : {mlp_exact_match:.4f}")


# ============================================================
# STEP 3: Per-root-cause evaluation
# ============================================================

print("\n" + "=" * 70)
print("PER ROOT-CAUSE PERFORMANCE")
print("=" * 70)

report = classification_report(
    y_test_mlp,
    y_pred_mlp,
    target_names=y_coolant.columns,
    zero_division=0,
    output_dict=True
)

per_cause_results = pd.DataFrame(report).T

display(
    per_cause_results[
        ["precision", "recall", "f1-score", "support"]
    ]
)


# ============================================================
# STEP 4: Create a compact model-results table
# ============================================================

mlp_results = pd.DataFrame({
    "Model": ["MLP"],
    "Hamming Loss": [mlp_hamming_loss],
    "Micro Precision": [mlp_precision],
    "Micro Recall": [mlp_recall],
    "Micro F1": [mlp_f1],
    "Macro F1": [mlp_macro_f1],
    "Exact Match": [mlp_exact_match]
})

print("\n" + "=" * 70)
print("MLP SUMMARY")
print("=" * 70)

display(mlp_results)


# ============================================================
# STEP 5: Display actual vs predicted for each test case
# ============================================================

case_results = []

for index in y_test.index:
    actual_causes = list(
        y_test.loc[index].index[
            y_test.loc[index] == 1
        ]
    )

    predicted_causes = list(
        prediction_df.loc[index].index[
            prediction_df.loc[index] == 1
        ]
    )

    case_results.append({
        "case_name":
            coolant_features.loc[
                index,
                "case_name"
            ],
        "actual_root_causes":
            actual_causes,
        "predicted_root_causes":
            predicted_causes,
        "correct":
            set(actual_causes) == set(predicted_causes)
    })

case_results_df = pd.DataFrame(case_results)

print("\n" + "=" * 70)
print("CASE-LEVEL RCA RESULTS")
print("=" * 70)

display(case_results_df)


## 2.5 Coolant 1D CNN Model

A 1D CNN is trained on the same standardized feature vector after reshaping it for convolution. Convolutional filters learn local patterns in the engineered feature representation, while pooling reduces dimensionality.

In [ ]:
# ============================================================
# COOLANT RCA - 1D CNN MODEL
# ============================================================

import tensorflow as tf
import numpy as np
import pandas as pd

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# STEP 1: Prepare data for CNN
# ============================================================

X_train_cnn = X_train_scaled.values
X_test_cnn = X_test_scaled.values

y_train_cnn = y_train.values
y_test_cnn = y_test.values


print("Original training shape:")
print(X_train_cnn.shape)

print("Original testing shape:")
print(X_test_cnn.shape)


# ============================================================
# STEP 2: Reshape for Conv1D
# ============================================================
# Conv1D expects:
#
# (samples, timesteps, channels)
#
# Here:
# samples  = number of fault cases
# timesteps = 195 extracted features
# channels = 1


X_train_cnn = X_train_cnn.reshape(
    X_train_cnn.shape[0],
    X_train_cnn.shape[1],
    1
)

X_test_cnn = X_test_cnn.reshape(
    X_test_cnn.shape[0],
    X_test_cnn.shape[1],
    1
)


print("\nCNN training shape:")
print(X_train_cnn.shape)

print("CNN testing shape:")
print(X_test_cnn.shape)

# Store final processed coolant CNN test data in globally accessible variables
X_test_coolant_cnn = X_test_cnn


# ============================================================
# STEP 3: Build 1D CNN
# ============================================================

cnn_coolant = Sequential([

    Input(
        shape=(X_train_cnn.shape[1], 1)
    ),

    Conv1D(
        filters=32,
        kernel_size=3,
        activation="relu",
        padding="same"
    ),

    MaxPooling1D(
        pool_size=2
    ),

    Dropout(0.25),

    Conv1D(
        filters=16,
        kernel_size=3,
        activation="relu",
        padding="same"
    ),

    MaxPooling1D(
        pool_size=2
    ),

    Dropout(0.25),

    Flatten(),

    Dense(
        32,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        6,
        activation="sigmoid"
    )
])


# ============================================================
# STEP 4: Compile CNN
# ============================================================

cnn_coolant.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="binary_crossentropy",

    metrics=[
        tf.keras.metrics.BinaryAccuracy(
            name="binary_accuracy"
        )
    ]
)


# ============================================================
# STEP 5: Display architecture
# ============================================================

print("\n" + "=" * 70)
print("CNN ARCHITECTURE")
print("=" * 70)

cnn_coolant.summary()


# ============================================================
# STEP 6: Early stopping
# ============================================================

early_stopping_cnn = EarlyStopping(

    monitor="val_loss",

    patience=15,

    restore_best_weights=True
)


# ============================================================
# STEP 7: Train CNN
# ============================================================

history_cnn = cnn_coolant.fit(

    X_train_cnn,

    y_train_cnn,

    epochs=100,

    batch_size=4,

    validation_split=0.20,

    callbacks=[early_stopping_cnn],

    verbose=1
)


# ============================================================
# STEP 8: Evaluate CNN on test data
# ============================================================

cnn_test_loss, cnn_test_accuracy = cnn_coolant.evaluate(

    X_test_cnn,

    y_test_cnn,

    verbose=0
)


print("\n" + "=" * 70)
print("CNN TEST RESULTS")
print("=" * 70)

print("Test Loss:", cnn_test_loss)

print(
    "Test Binary Accuracy:",
    cnn_test_accuracy
)


# ============================================================
# STEP 9: Generate root-cause probabilities
# ============================================================

y_pred_cnn_probability = cnn_coolant.predict(

    X_test_cnn,

    verbose=0
)


# ============================================================
# STEP 10: Convert probabilities to binary predictions
# ============================================================

threshold = 0.5

y_pred_cnn = (
    y_pred_cnn_probability >= threshold
).astype(int)


# ============================================================
# STEP 11: Display probabilities
# ============================================================

print("\n" + "=" * 70)
print("CNN ROOT-CAUSE PROBABILITIES")
print("=" * 70)

cnn_probability_df = pd.DataFrame(

    y_pred_cnn_probability,

    columns=y_coolant.columns,

    index=y_test.index
)

display(cnn_probability_df)


# ============================================================
# STEP 12: Display predicted root causes
# ============================================================

print("\n" + "=" * 70)
print("CNN PREDICTED ROOT CAUSES")
print("=" * 70)

cnn_prediction_df = pd.DataFrame(

    y_pred_cnn,

    columns=y_coolant.columns,

    index=y_test.index
)

display(cnn_prediction_df)


# ============================================================
# STEP 13: Actual vs predicted root causes
# ============================================================

cnn_case_results = []

for index in y_test.index:

    actual_causes = list(
        y_test.loc[index].index[
            y_test.loc[index] == 1
        ]
    )

    predicted_causes = list(
        cnn_prediction_df.loc[index].index[
            cnn_prediction_df.loc[index] == 1
        ]
    )

    cnn_case_results.append({

        "case_name":
            coolant_features.loc[
                index,
                "case_name"
            ],

        "actual_root_causes":
            actual_causes,

        "predicted_root_causes":
            predicted_causes,

        "correct":
            set(actual_causes)
            == set(predicted_causes)
    })


cnn_case_results_df = pd.DataFrame(
    cnn_case_results
)

print("\n" + "=" * 70)
print("CNN CASE-LEVEL RCA RESULTS")
print("=" * 70)

display(cnn_case_results_df)

## 2.6 Coolant Model Evaluation

The MLP and 1D CNN predictions are evaluated and compared using consistent multi-label performance metrics.

In [ ]:
# ============================================================
# COOLANT RCA - MLP vs CNN COMPARISON
# ============================================================

from sklearn.metrics import (
    hamming_loss,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)
import pandas as pd
import numpy as np


# ============================================================
# STEP 1: Calculate MLP metrics
# ============================================================

mlp_metrics = {
    "Model": "MLP",
    "Hamming Loss": hamming_loss(
        y_test_mlp,
        y_pred_mlp
    ),
    "Micro Precision": precision_score(
        y_test_mlp,
        y_pred_mlp,
        average="micro",
        zero_division=0
    ),
    "Micro Recall": recall_score(
        y_test_mlp,
        y_pred_mlp,
        average="micro",
        zero_division=0
    ),
    "Micro F1": f1_score(
        y_test_mlp,
        y_pred_mlp,
        average="micro",
        zero_division=0
    ),
    "Macro F1": f1_score(
        y_test_mlp,
        y_pred_mlp,
        average="macro",
        zero_division=0
    ),
    "Exact Match": accuracy_score(
        y_test_mlp,
        y_pred_mlp
    )
}


# ============================================================
# STEP 2: Calculate CNN metrics
# ============================================================

cnn_metrics = {
    "Model": "1D CNN",
    "Hamming Loss": hamming_loss(
        y_test_cnn,
        y_pred_cnn
    ),
    "Micro Precision": precision_score(
        y_test_cnn,
        y_pred_cnn,
        average="micro",
        zero_division=0
    ),
    "Micro Recall": recall_score(
        y_test_cnn,
        y_pred_cnn,
        average="micro",
        zero_division=0
    ),
    "Micro F1": f1_score(
        y_test_cnn,
        y_pred_cnn,
        average="macro",
        zero_division=0
    ),
    "Macro F1": f1_score(
        y_test_cnn,
        y_pred_cnn,
        average="macro",
        zero_division=0
    ),
    "Exact Match": accuracy_score(
        y_test_cnn,
        y_pred_cnn
    )
}


# ============================================================
# STEP 3: Create comparison table
# ============================================================

model_comparison = pd.DataFrame([
    mlp_metrics,
    cnn_metrics
])


print("=" * 80)
print("COOLANT RCA - MLP vs 1D CNN")
print("=" * 80)

display(
    model_comparison.round(4)
)


# ============================================================
# STEP 4: Per-root-cause F1 comparison
# ============================================================

mlp_f1_per_cause = f1_score(
    y_test_mlp,
    y_pred_mlp,
    average=None,
    zero_division=0
)

cnn_f1_per_cause = f1_score(
    y_test_cnn,
    y_pred_cnn,
    average=None,
    zero_division=0
)


per_cause_comparison = pd.DataFrame({
    "Root Cause": y_coolant.columns,
    "MLP F1": mlp_f1_per_cause,
    "CNN F1": cnn_f1_per_cause
})


print("\n" + "=" * 80)
print("PER ROOT-CAUSE F1 COMPARISON")
print("=" * 80)

display(
    per_cause_comparison.round(4)
)


# ============================================================
# STEP 5: Determine better model
# ============================================================

mlp_f1_value = mlp_metrics["Micro F1"]
cnn_f1_value = cnn_metrics["Micro F1"]

print("\n" + "=" * 80)
print("BEST MODEL")
print("=" * 80)

if mlp_f1_value > cnn_f1_value:
    print("MLP has the higher Micro F1-score.")
elif cnn_f1_value > mlp_f1_value:
    print("1D CNN has the higher Micro F1-score.")
else:
    print("MLP and 1D CNN have the same Micro F1-score.")

print("\nImportant:")
print("Lower Hamming Loss = better")
print("Higher Precision   = better")
print("Higher Recall      = better")
print("Higher F1          = better")
print("Higher Exact Match = better")


# 3. Hydraulics Root Cause Analysis

## 3.1 Hydraulics Feature Extraction

This cell converts each Hydraulics fault dataset into a fixed-length feature vector using before/after fault-window statistics and process-state features. Metadata is retained only for case alignment and is excluded from model inputs later.

In [ ]:
# ============================================================
# HYDRAULICS RCA - FEATURE EXTRACTION
# ============================================================

import glob
import json
import os
import numpy as np
import pandas as pd


# ============================================================
# STEP 1: Find all hydraulics fault CSV files
# ============================================================

hydraulic_files = glob.glob(
    "causrca_data/dig_twin/exp_hydraulics/*/run_*/faultDataset_*.csv"
)

print("Number of hydraulics fault files:", len(hydraulic_files))


# ============================================================
# STEP 2: Feature extraction function
# ============================================================

def extract_hydraulics_features(file, window=30):

    # --------------------------------------------------------
    # Read signal data
    # --------------------------------------------------------

    df = pd.read_csv(file)

    # Read corresponding causes.json
    run_folder = os.path.dirname(file)
    causes_file = os.path.join(
        run_folder,
        "causes.json"
    )

    with open(causes_file, "r") as f:
        causes = json.load(f)

    cause_start = causes["cause_start_at"]

    # --------------------------------------------------------
    # Identify case and experiment
    # --------------------------------------------------------

    run_name = os.path.basename(run_folder)
    exp_folder = os.path.dirname(run_folder)
    exp_name = os.path.basename(exp_folder)

    case_name = f"{exp_name}/{run_name}"

    # --------------------------------------------------------
    # Check and standardize time column
    # --------------------------------------------------------

    # Define possible time column names, including 'time_s'
    possible_time_columns = [
        "time_s", # Added 'time_s' first as it was identified in the error
        "time",
        "Time",
        "timestamp",
        "Timestamp",
        "Time_s",
        "timestamp_s",
        "Timestamp_s",
        "relative_time",
        "RelativeTime"
    ]

    found_time_column = None
    for col in possible_time_columns:
        if col in df.columns:
            found_time_column = col
            break

    if found_time_column is None:
        # Fallback: search for a column containing 'time' or 'timestamp' (case-insensitive)
        for col in df.columns:
            if "time" in str(col).lower() or "timestamp" in str(col).lower():
                found_time_column = col
                break

    if found_time_column is None:
        raise KeyError(
            f"'time' column not found in {file}. "
            f"Available columns: {list(df.columns)}"
        )

    # Rename the found time column to 'time' for consistent processing
    if found_time_column != "time":
        df = df.rename(columns={found_time_column: "time"})

    # --------------------------------------------------------
    # Convert time to numeric
    # --------------------------------------------------------

    df["time"] = pd.to_numeric(
        df["time"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["time"]
    )

    # --------------------------------------------------------
    # Create BEFORE and AFTER windows
    # --------------------------------------------------------

    before = df[
        (df["time"] >= cause_start - window) &
        (df["time"] < cause_start)
    ].copy()

    after = df[
        (df["time"] >= cause_start) &
        (df["time"] <= cause_start + window)
    ].copy()

    # --------------------------------------------------------
    # Containers
    # --------------------------------------------------------

    continuous_features = {}
    binary_features = {}
    counter_features = {}
    categorical_features = {}
    alarm_features = {}

    # ========================================================
    # STEP 3: Process each node
    # ========================================================

    for node in df["node"].dropna().unique():

        node_before = before[
            before["node"] == node
        ]

        node_after = after[
            after["node"] == node
        ]

        if len(node_before) == 0 and len(node_after) == 0:
            continue

        # ----------------------------------------------------
        # Determine data type
        # ----------------------------------------------------

        node_type = None

        if "type" in df.columns:

            type_values = df.loc[
                df["node"] == node,
                "type"
            ].dropna().astype(str)

            if len(type_values) > 0:
                node_type = type_values.iloc[0].lower()

        # ----------------------------------------------------
        # Values
        # ----------------------------------------------------

        before_values = pd.to_numeric(
            node_before["value"],
            errors="coerce"
        ).dropna()

        after_values = pd.to_numeric(
            node_after["value"],
            errors="coerce"
        ).dropna()

        # ====================================================
        # CONTINUOUS
        # ====================================================

        if node_type == "continuous":

            if len(before_values) > 0:

                before_mean = before_values.mean()
                before_std = before_values.std()

            else:

                before_mean = np.nan
                before_std = np.nan

            if len(after_values) > 0:

                after_mean = after_values.mean()
                after_std = after_values.std()

            else:

                after_mean = np.nan
                after_std = np.nan

            mean_change = (
                after_mean - before_mean
            )

            absolute_change = abs(
                mean_change
            )

            if (
                pd.notna(before_mean)
                and before_mean != 0
            ):

                relative_change = (
                    mean_change /
                    abs(before_mean)
                )

            else:

                relative_change = np.nan

            std_change = (
                after_std - before_std
            )

            continuous_features[
                f"{node}_before_mean"
            ] = before_mean

            continuous_features[
                f"{node}_before_std"
            ] = before_std

            continuous_features[
                f"{node}_after_mean"
            ] = after_mean

            continuous_features[
                f"{node}_after_std"
            ] = after_std

            continuous_features[
                f"{node}_mean_change"
            ] = mean_change

            continuous_features[
                f"{node}_absolute_change"
            ] = absolute_change

            continuous_features[
                f"{node}_relative_change"
            ] = relative_change

            continuous_features[
                f"{node}_std_change"
            ] = std_change

        # ====================================================
        # BINARY
        # ====================================================

        elif node_type == "binary":

            before_binary = pd.to_numeric(
                node_before["value"],
                errors="coerce"
            ).dropna()

            after_binary = pd.to_numeric(
                node_after["value"],
                errors="coerce"
            ).dropna()

            before_ratio = (
                before_binary.mean()
                if len(before_binary) > 0
                else np.nan
            )

            after_ratio = (
                after_binary.mean()
                if len(after_binary) > 0
                else np.nan
            )

            transitions = 0

            combined_values = pd.concat(
                [
                    before_binary,
                    after_binary
                ]
            )

            if len(combined_values) > 1:

                transitions = (
                    combined_values
                    .diff()
                    .abs()
                    .fillna(0)
                    .gt(0)
                    .sum()
                )

            binary_features[
                f"{node}_before_ratio"
            ] = before_ratio

            binary_features[
                f"{node}_after_ratio"
            ] = after_ratio

            binary_features[
                f"{node}_ratio_change"
            ] = after_ratio - before_ratio

            binary_features[
                f"{node}_transitions"
            ] = transitions

        # ====================================================
        # COUNTER
        # ====================================================

        elif node_type == "counter":

            if len(before_values) > 0:

                before_mean = before_values.mean()

            else:

                before_mean = np.nan

            if len(after_values) > 0:

                after_mean = after_values.mean()

            else:

                after_mean = np.nan

            change = (
                after_mean - before_mean
            )

            before_rate = np.nan
            after_rate = np.nan

            if len(node_before) > 1:

                time_range = (
                    node_before["time"].max()
                    - node_before["time"].min()
                )

                if time_range > 0:

                    before_rate = (
                        before_values.iloc[-1]
                        - before_values.iloc[0]
                    ) / time_range

            if len(node_after) > 1:

                time_range = (
                    node_after["time"].max()
                    - node_after["time"].min()
                )

                if time_range > 0:

                    after_rate = (
                        after_values.iloc[-1]
                        - after_values.iloc[0]
                    ) / time_range

            counter_features[
                f"{node}_before_mean"
            ] = before_mean

            counter_features[
                f"{node}_after_mean"
            ] = after_mean

            counter_features[
                f"{node}_change"
            ] = change

            counter_features[
                f"{node}_before_rate"
            ] = before_rate

            counter_features[
                f"{node}_after_rate"
            ] = after_rate

            counter_features[
                f"{node}_rate_difference"
            ] = after_rate - before_rate

        # ====================================================
        # CATEGORICAL
        # ====================================================

        elif node_type == "categorical":

            before_categories = (
                node_before["value"]
                .astype(str)
                .value_counts(
                    normalize=True
                )
            )

            after_categories = (
                node_after["value"]
                .astype(str)
                .value_counts(
                    normalize=True
                )
            )

            categories = set(
                before_categories.index
            ).union(
                set(after_categories.index)
            )

            for category in categories:

                before_prop = (
                    before_categories.get(
                        category,
                        0
                    )
                )

                after_prop = (
                    after_categories.get(
                        category,
                        0
                    )
                )

                categorical_features[
                    f"{node}_before_{category}"
                ] = before_prop

                categorical_features[
                    f"{node}_after_{category}"
                ] = after_prop

                categorical_features[
                    f"{node}_change_{category}"
                ] = (
                    after_prop - before_prop
                )

        # ====================================================
        # ALARM
        # ====================================================

        elif node_type == "alarm":

            before_alarm = pd.to_numeric(
                node_before["value"],
                errors="coerce"
            ).dropna()

            after_alarm = pd.to_numeric(
                node_after["value"],
                errors="coerce"
            ).dropna()

            before_ratio = (
                before_alarm.mean()
                if len(before_alarm) > 0
                else 0
            )

            after_ratio = (
                after_alarm.mean()
                if len(after_alarm) > 0
                else 0
            )

            alarm_features[
                f"{node}_before_ratio"
            ] = before_ratio

            alarm_features[
                f"{node}_after_ratio"
            ] = after_ratio

            alarm_features[
                f"{node}_ratio_change"
            ] = (
                after_ratio - before_ratio
            )

            # First alarm after cause start
            alarm_after = node_after[
                pd.to_numeric(
                    node_after["value"],
                    errors="coerce"
                ) > 0
            ]

            if len(alarm_after) > 0:

                first_alarm_time = (
                    alarm_after["time"].min()
                )

                alarm_delay = (
                    first_alarm_time
                    - cause_start
                )

            else:

                alarm_delay = np.nan

            alarm_features[
                f"{node}_first_alarm_delay"
            ] = alarm_delay

    # ========================================================
    # STEP 4: Combine all features
    # ========================================================

    all_features = {}

    all_features.update(
        continuous_features
    )

    all_features.update(
        binary_features
    )

    all_features.update(
        counter_features
    )

    all_features.update(
        categorical_features
    )

    all_features.update(
        alarm_features
    )

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    all_features["case_name"] = case_name
    all_features["experiment"] = exp_name
    all_features["run"] = run_name
    all_features["cause_start_at"] = cause_start

    return pd.DataFrame([all_features])


# ============================================================
# STEP 5: Extract features from all hydraulics cases
# ============================================================

hydraulics_feature_list = []

for file in hydraulic_files:

    try:

        features = extract_hydraulics_features(
            file,
            window=30
        )

        hydraulics_feature_list.append(
            features
        )

    except Exception as e:

        print(
            "\nERROR:",
            file
        )

        print(
            str(e)
        )


# ============================================================
# STEP 6: Combine all cases
# ============================================================

hydraulics_features = pd.concat(
    hydraulics_feature_list,
    ignore_index=True
)


# ============================================================
# STEP 7: Display results
# ============================================================

print("\n" + "=" * 70)
print("HYDRAULICS FEATURE EXTRACTION")
print("=" * 70)

print(
    "\nFeature matrix shape:",
    hydraulics_features.shape
)

print(
    "\nNumber of cases:",
    len(hydraulics_features)
)

print(
    "\nNumber of features:",
    hydraulics_features.shape[1] - 4
)

print(
    "\nMissing values:",
    hydraulics_features.isna().sum().sum()
)

print(
    "\nInfinite values:",
    np.isinf(
        hydraulics_features
        .select_dtypes(include=np.number)
    ).sum().sum()
)

print("\nFirst 5 cases:")

display(
    hydraulics_features.head()
)


## 3.2 Hydraulics Feature Reduction

Features with excessive missingness are removed to reduce sparsity. The resulting feature matrix preserves informative process variables while reducing unreliable dimensions.

In [ ]:
# ============================================================
# HYDRAULICS RCA - FEATURE REDUCTION
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# STEP 1: Separate metadata from ML features
# ============================================================

metadata_cols = [
    "case_name",
    "experiment",
    "run",
    "cause_start_at"
]

feature_cols = [
    col
    for col in hydraulics_features.columns
    if col not in metadata_cols
]

X_hydraulics = hydraulics_features[
    feature_cols
].copy()


# ============================================================
# STEP 2: Check original feature matrix
# ============================================================

print("=" * 70)
print("HYDRAULICS FEATURE REDUCTION")
print("=" * 70)

print("\nOriginal feature matrix:")
print(X_hydraulics.shape)

print(
    "Original missing values:",
    X_hydraulics.isna().sum().sum()
)


# ============================================================
# STEP 3: Calculate missing percentage
# ============================================================

missing_percentage = (
    X_hydraulics.isna().mean() * 100
).sort_values(
    ascending=False
)


print("\nTop 20 features by missing percentage:")

display(
    missing_percentage.head(20).to_frame(
        "Missing %"
    )
)


# ============================================================
# STEP 4: Remove features with >80% missing values
# ============================================================

missing_threshold = 80

keep_features = (
    missing_percentage[
        missing_percentage <= missing_threshold
    ]
    .index
)


X_hydraulics_reduced = X_hydraulics[
    keep_features
].copy()


# ============================================================
# STEP 5: Display reduction results
# ============================================================

features_removed = (
    X_hydraulics.shape[1]
    -
    X_hydraulics_reduced.shape[1]
)

remaining_missing = (
    X_hydraulics_reduced
    .isna()
    .sum()
    .sum()
)


remaining_missing_percentage = (
    remaining_missing
    /
    (
        X_hydraulics_reduced.shape[0]
        *
        X_hydraulics_reduced.shape[1]
    )
) * 100


print("\n" + "=" * 70)
print("FEATURE REDUCTION RESULTS")
print("=" * 70)

print(
    "\nOriginal features:",
    X_hydraulics.shape[1]
)

print(
    "Features removed:",
    features_removed
)

print(
    "Remaining features:",
    X_hydraulics_reduced.shape[1]
)

print(
    "Remaining missing values:",
    remaining_missing
)

print(
    "Remaining missing percentage:",
    round(
        remaining_missing_percentage,
        2
    ),
    "%"
)


# ============================================================
# STEP 6: Final matrix
# ============================================================

print("\nFinal hydraulics feature matrix:")

print(
    X_hydraulics_reduced.shape
)


## 3.3 Hydraulics Multi-Label Ground Truth

Root causes are extracted and aligned with Hydraulics cases. `MultiLabelBinarizer` converts simultaneous root causes into a binary multi-label target matrix.

In [ ]:
# ============================================================
# HYDRAULICS RCA - GROUND-TRUTH MULTI-LABEL TARGET
# ============================================================

import glob
import json
import os
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer


# ============================================================
# STEP 1: Read all hydraulics experiment descriptions
# ============================================================

description_files = glob.glob(
    "causrca_data/dig_twin/exp_hydraulics/*/*_description.json"
)

print("=" * 70)
print("HYDRAULICS GROUND-TRUTH EXTRACTION")
print("=" * 70)

print(
    "\nNumber of experiment descriptions:",
    len(description_files)
)


# ============================================================
# STEP 2: Create Experiment -> Root Causes mapping
# ============================================================

experiment_root_causes = {}

for file in description_files:
    with open(file, "r") as f:
        description = json.load(f)
        experiment_id = description["exp_id"]
        root_causes = description.get(
            "manipulatedVars",
            []
        )
        experiment_root_causes[
            experiment_id
        ] = root_causes


# ============================================================
# STEP 3: Display ground-truth mapping
# ============================================================

print("\n" + "=" * 70)
print("EXPERIMENT → ROOT CAUSES")
print("=" * 70)

for experiment, causes in sorted(
    experiment_root_causes.items()
):
    print(
        f"\n{experiment}: {causes}"
    )


# ============================================================
# STEP 4: Match each feature row to its experiment
# ============================================================

root_causes_per_case = []

matched_cases = 0
unmatched_cases = 0
unmatched_experiments = []


for experiment in hydraulics_features[
    "experiment"
]:
    if experiment in experiment_root_causes:
        root_causes_per_case.append(
            experiment_root_causes[
                experiment
            ]
        )
        matched_cases += 1
    else:
        root_causes_per_case.append([])
        unmatched_cases += 1
        unmatched_experiments.append(
            experiment
        )


# ============================================================
# STEP 5: Check matching
# ============================================================

print("\n" + "=" * 70)
print("CASE MATCHING")
print("=" * 70)

print(
    "\nMatched cases:",
    matched_cases
)

print(
    "Unmatched cases:",
    unmatched_cases
)


if unmatched_cases > 0:
    print(
        "\nUnmatched experiments:"
    )
    print(
        sorted(
            set(unmatched_experiments)
        )
    )


# ============================================================
# STEP 6: Stop if matching failed
# ============================================================

if matched_cases == 0:
    raise ValueError(
        "No hydraulics cases were matched "
        "with experiment descriptions."
    )


# ============================================================
# STEP 7: Convert root causes to binary labels
# ============================================================

mlb_hydraulics = MultiLabelBinarizer()

y_hydraulics = pd.DataFrame(
    mlb_hydraulics.fit_transform(
        root_causes_per_case
    ),
    columns=mlb_hydraulics.classes_,
    index=hydraulics_features.index
)


# ============================================================
# STEP 8: Display target information
# ============================================================

print("\n" + "=" * 70)
print("HYDRAULICS TARGET")
print("=" * 70)

print(
    "\nRoot-cause classes:"
)

print(
    list(
        y_hydraulics.columns
    )
)

print(
    "\nTarget shape:"
)

print(
    y_hydraulics.shape
)


print(
    "\nRoot-cause labels:"
)

display(
    y_hydraulics
)


# ============================================================
# STEP 9: Count cases containing each root cause
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "ROOT-CAUSE DISTRIBUTION"
)

print(
    "=" * 70
)

root_cause_counts = (
    y_hydraulics
    .sum()
    .sort_values(
        ascending=False
    )
)

display(
    root_cause_counts.to_frame(
        "Number of Cases"
    )
)


# ============================================================
# STEP 10: Check X and y alignment
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "X AND y ALIGNMENT CHECK"
)

print(
    "=" * 70
)


if (
    len(hydraulics_features)
    == len(y_hydraulics)
    and
    all(
        hydraulics_features.index
        ==
        y_hydraulics.index
    )
):

    print(
        "\nX and y row check: PASSED"
    )

else:

    raise ValueError(
        "X and y are not aligned."
    )


# ============================================================
# STEP 11: Create final feature matrix
# ============================================================

X_hydraulics_final = (
    X_hydraulics_reduced.copy()
)


print(
    "\nFinal X shape:",
    X_hydraulics_final.shape
)

print(
    "Final y shape:",
    y_hydraulics.shape
)

## 3.4 Hydraulics Preprocessing

The reduced feature matrix is split by case into training and test sets. Median imputation handles remaining missing values, and standardization places numerical features on a comparable scale before neural-network training.

In [ ]:
# ============================================================
# HYDRAULICS RCA - TRAIN/TEST SPLIT + IMPUTATION + SCALING
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np


# ============================================================
# STEP 1: Verify X and y
# ============================================================

print("=" * 70)
print("HYDRAULICS PREPROCESSING")
print("=" * 70)

print("\nX shape:", X_hydraulics_final.shape)
print("y shape:", y_hydraulics.shape)


if len(X_hydraulics_final) != len(y_hydraulics):

    raise ValueError(
            "X and y have different numbers of rows."
        )


# ============================================================
# STEP 2: Split by CASE
# ============================================================
# We split using case names so that the same experiment/run
# cannot accidentally appear in both training and testing.

case_names = hydraulics_features[
    "case_name"
].values


train_cases, test_cases = train_test_split(
    case_names,
    test_size=0.20,
    random_state=42
)


train_mask = hydraulics_features[
    "case_name"
].isin(train_cases)

test_mask = hydraulics_features[
    "case_name"
].isin(test_cases)


X_train = X_hydraulics_final.loc[
    train_mask
].copy()

X_test = X_hydraulics_final.loc[
    test_mask
].copy()

y_train = y_hydraulics.loc[
    train_mask
].copy()

y_test = y_hydraulics.loc[
    test_mask
].copy()


# ============================================================
# STEP 3: Verify split
# ============================================================

print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print("\nTraining cases:", len(train_cases))
print("Testing cases :", len(test_cases))

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_test :", y_test.shape)


# ============================================================
# STEP 4: Check root causes in training/testing
# ============================================================

print("\n" + "=" * 70)
print("ROOT-CAUSE DISTRIBUTION")
print("=" * 70)

print("\nTraining set:")
print(
    y_train.sum()
    .sort_values(
        ascending=False
    )
)

print("\nTesting set:")
print(
    y_test.sum()
    .sort_values(
        ascending=False
    )
)


# ============================================================
# STEP 5: Check whether every root cause exists in training
# ============================================================

missing_training_classes = [
    col
    for col in y_train.columns
    if y_train[col].sum() == 0
]


if len(missing_training_classes) > 0:

    print(
            "\nWARNING: These root causes are absent "
            "from the training set:"
        )

    print(
            missing_training_classes
        )

else:

    print(
            "\nAll root causes are present "
            "in the training set."
        )


# ============================================================
# STEP 6: Imputation
# ============================================================
# IMPORTANT:
# The median is learned ONLY from the training set.
# The test set uses the training-set medians.
#
# This prevents information from the test set leaking
# into the training process.

imputer = SimpleImputer(
    strategy="median"
)


X_train_imputed = imputer.fit_transform(
    X_train
)

X_test_imputed = imputer.transform(
    X_test
)


# ============================================================
# STEP 7: Check imputation dimensions
# ============================================================

print("\n" + "=" * 70)
print("AFTER IMPUTATION")
print("=" * 70)

print(
    "\nX_train_imputed:",
    X_train_imputed.shape
)

print(
    "X_test_imputed :",
    X_test_imputed.shape
)

print(
    "\nTraining missing values:",
    np.isnan(
        X_train_imputed
    ).sum()
)

print(
    "Testing missing values:",
    np.isnan(
        X_test_imputed
    ).sum()
)


# ============================================================
# STEP 8: Convert back to DataFrames
# ============================================================

imputed_columns = X_train.columns


X_train_imputed = pd.DataFrame(
    X_train_imputed,
    columns=imputed_columns,
    index=X_train.index
)

X_test_imputed = pd.DataFrame(
    X_test_imputed,
    columns=imputed_columns,
    index=X_test.index
)


# ============================================================
# STEP 9: Standardization
# ============================================================
# StandardScaler learns mean and standard deviation ONLY
# from the training set.

scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(
    X_train_imputed
)

X_test_scaled = scaler.transform(
    X_test_imputed
)


# ============================================================
# STEP 10: Convert scaled arrays to DataFrames
# ============================================================

X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=imputed_columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled,
    columns=imputed_columns,
    index=X_test.index
)


# ============================================================
# STEP 11: Final checks
# ============================================================

print("\n" + "=" * 70)
print("FINAL PREPROCESSED HYDRAULICS DATA")
print("=" * 70)

print(
    "\nX_train:",
    X_train_scaled.shape
)

print(
    "X_test :",
    X_test_scaled.shape
)

print(
    "\ny_train:",
    y_train.shape
)

print(
    "y_test :",
    y_test.shape
)


print(
    "\nTraining NaN:",
    X_train_scaled.isna().sum().sum()
)

print(
    "Testing NaN:",
    X_test_scaled.isna().sum().sum()
)


print(
    "\nTraining mean:",
    round(
        X_train_scaled.values.mean(),
        5
    )
)

print(
    "Training standard deviation:",
    round(
        X_train_scaled.values.std(),
        5
    )
)


# ============================================================
# STEP 12: Verify final row alignment
# ============================================================

if (
    list(X_train_scaled.index)
    ==
    list(y_train.index)
    and
    list(X_test_scaled.index)
    ==
    list(y_test.index)
):

    print(
            "\nX and y row alignment: PASSED"
        )

else:

    raise ValueError(
            "X and y row alignment failed."
        )


# ============================================================
# STEP 13: Store final arrays for TensorFlow
# ============================================================

X_train_hydraulics = X_train_scaled.values
X_test_hydraulics_final = X_test_scaled.values # Renamed for global access

y_train_hydraulics = y_train.values.astype(
    "float32"
)

y_test_hydraulics_final = y_test.values.astype(
    "float32"
) # Renamed for global access


print("\n" + "=" * 70)
print("READY FOR HYDRAULICS DEEP LEARNING")
print("=" * 70)

print(
    "\nX_train_hydraulics:",
    X_train_hydraulics.shape
)

print(
    "X_test_hydraulics_final:",
    X_test_hydraulics_final.shape
)

print(
    "y_train_hydraulics:",
    y_train_hydraulics.shape
)

print(
    "y_test_hydraulics_final:",
    y_test_hydraulics_final.shape
)

## 3.5 Hydraulics MLP and 1D CNN Models

Both deep learning architectures are trained independently on the same preprocessed Hydraulics data. The MLP models global feature interactions, while the 1D CNN learns local feature patterns.

In [ ]:
# ============================================================
# HYDRAULICS RCA - MLP AND 1D CNN
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    Dropout,
    Conv1D,
    MaxPooling1D,
    Flatten,
    BatchNormalization
)
from tensorflow.keras.optimizers import Adam


# ============================================================
# STEP 1: Reproducibility
# ============================================================

np.random.seed(42)
tf.random.set_seed(42)


# ============================================================
# STEP 2: Display dataset information
# ============================================================

print("=" * 70)
print("HYDRAULICS RCA - DEEP LEARNING")
print("=" * 70)

print("\nTraining features:",
      X_train_hydraulics.shape)

print("Testing features :",
      X_test_hydraulics_final.shape)

print("\nTraining labels:",
      y_train_hydraulics.shape)

print("Testing labels :",
      y_test_hydraulics_final.shape)

print("\nRoot-cause classes:")
print(list(y_hydraulics.columns))


# ============================================================
# STEP 3: Number of input features and output classes
# ============================================================

n_features = X_train_hydraulics.shape[1]

n_classes = y_train_hydraulics.shape[1]


# ============================================================
# ============================================================
# MODEL 1: MLP
# ============================================================
# ============================================================

print("\n" + "=" * 70)
print("BUILDING MLP")
print("=" * 70)


mlp_hydraulics = Sequential([

    Dense(
        128,
        activation="relu",
        input_shape=(n_features,)
    ),

    BatchNormalization(),

    Dropout(0.30),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        n_classes,
        activation="sigmoid"
    )
])


# ============================================================
# Compile MLP
# ============================================================

mlp_hydraulics.compile(

    optimizer=Adam(
        learning_rate=0.001
    ),

    loss="binary_crossentropy"
)


print("\nMLP architecture:")

mlp_hydraulics.summary()


# ============================================================
# STEP 4: Train MLP
# ============================================================

print("\n" + "=" * 70)
print("TRAINING MLP")
print("=" * 70)


mlp_history = mlp_hydraulics.fit(

    X_train_hydraulics,

    y_train_hydraulics,

    epochs=50,

    batch_size=8,

    validation_split=0.20,

    shuffle=True,

    verbose=1
)


# ============================================================
# ============================================================
# MODEL 2: 1D CNN
# ============================================================
# ============================================================

print("\n" + "=" * 70)
print("BUILDING 1D CNN")
print("=" * 70)


# ------------------------------------------------------------
# CNN requires 3D input:
#
# (samples, features, channels)
#
# Current:
# (samples, features)
#
# Converted to:
# (samples, features, 1)
# ------------------------------------------------------------

X_train_cnn = X_train_hydraulics.reshape(
    X_train_hydraulics.shape[0],
    X_train_hydraulics.shape[1],
    1
)

X_test_cnn = X_test_hydraulics_final.reshape(
    X_test_hydraulics_final.shape[0],
    X_test_hydraulics_final.shape[1],
    1
)

# Store final processed hydraulics CNN test data in globally accessible variables
X_test_hydraulics_cnn = X_test_cnn

print("\nCNN training input:",
      X_train_cnn.shape)

print("CNN testing input :",
      X_test_cnn.shape)


# ============================================================
# Build CNN
# ============================================================

cnn_hydraulics = Sequential([

    Conv1D(
        filters=32,
        kernel_size=3,
        activation="relu",
        input_shape=(n_features, 1)
    ),

    BatchNormalization(),

    MaxPooling1D(
        pool_size=2
    ),

    Dropout(0.25),

    Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu"
    ),

    MaxPooling1D(
        pool_size=2
    ),

    Dropout(0.25),

    Flatten(),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.30),

    Dense(
        n_classes,
        activation="sigmoid"
    )
])


# ============================================================
# Compile CNN
# ============================================================

cnn_hydraulics.compile(

    optimizer=Adam(
        learning_rate=0.001
    ),

    loss="binary_crossentropy"
)


print("\n1D CNN architecture:")

cnn_hydraulics.summary()


# ============================================================
# STEP 5: Train CNN
# ============================================================

print("\n" + "=" * 70)
print("TRAINING 1D CNN")
print("=" * 70)


cnn_history = cnn_hydraulics.fit(

    X_train_cnn,

    y_train_hydraulics,

    epochs=50,

    batch_size=8,

    validation_split=0.20,

    shuffle=True,

    verbose=1
)


# ============================================================
# STEP 6: Generate predictions
# ============================================================

print("\n" + "=" * 70)
print("GENERATING PREDICTIONS")
print("=" * 70)


mlp_probabilities = mlp_hydraulics.predict(
    X_test_hydraulics_final,
    verbose=0
)

cnn_probabilities = cnn_hydraulics.predict(
    X_test_cnn,
    verbose=0
)


# ============================================================
# STEP 7: Convert probabilities to binary predictions
# ============================================================
# Threshold = 0.5
#
# probability >= 0.5 -> root cause present
# probability < 0.5  -> root cause absent
# ============================================================

mlp_predictions = (
    mlp_probabilities >= 0.5
).astype(int)


cnn_predictions = (
    cnn_probabilities >= 0.5
).astype(int)


# ============================================================
# STEP 8: Display predictions
# ============================================================

print("\nMLP predictions:")

display(
    pd.DataFrame(
        mlp_predictions,
        columns=y_hydraulics.columns
    )
)


print("\nCNN predictions:")

display(
    pd.DataFrame(
        cnn_predictions,
        columns=y_hydraulics.columns
    )
)


# ============================================================
# STEP 9: Display prediction probabilities
# ============================================================

print("\nMLP prediction probabilities:")

display(
    pd.DataFrame(
        np.round(
            mlp_probabilities,
            3
        ),
        columns=y_hydraulics.columns
    )
)


print("\nCNN prediction probabilities:")

display(
    pd.DataFrame(
        np.round(
            cnn_probabilities,
            3
        ),
        columns=y_hydraulics.columns
    )
)


# ============================================================
# STEP 10: Display actual root causes
# ============================================================

print("\nActual test-set root causes:")

display(
    y_test.reset_index(
        drop=True
    )
)


print("\n" + "=" * 70)
print("MLP AND CNN TRAINING COMPLETE")
print("=" * 70)

## 3.6 Hydraulics Model Evaluation

This cell evaluates the Hydraulics MLP and CNN separately using multi-label classification metrics and provides a direct comparison.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    hamming_loss,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)


# ============================================================
# STEP 1: Evaluate MLP Model
# ============================================================

mlp_metrics_hydraulics = {
    "Model": "MLP Hydraulics",
    "Hamming Loss": hamming_loss(
        y_test_hydraulics_final,
        mlp_predictions
    ),
    "Micro Precision": precision_score(
        y_test_hydraulics_final,
        mlp_predictions,
        average="micro",
        zero_division=0
    ),
    "Micro Recall": recall_score(
        y_test_hydraulics_final,
        mlp_predictions,
        average="micro",
        zero_division=0
    ),
    "Micro F1": f1_score(
        y_test_hydraulics_final,
        mlp_predictions,
        average="micro",
        zero_division=0
    ),
    "Macro F1": f1_score(
        y_test_hydraulics_final,
        mlp_predictions,
        average="macro",
        zero_division=0
    ),
    "Exact Match": accuracy_score(
        y_test_hydraulics_final,
        mlp_predictions
    )
}


# ============================================================
# STEP 2: Evaluate 1D CNN Model
# ============================================================

cnn_metrics_hydraulics = {
    "Model": "1D CNN Hydraulics",
    "Hamming Loss": hamming_loss(
        y_test_hydraulics_final,
        cnn_predictions
    ),
    "Micro Precision": precision_score(
        y_test_hydraulics_final,
        cnn_predictions,
        average="micro",
        zero_division=0
    ),
    "Micro Recall": recall_score(
        y_test_hydraulics_final,
        cnn_predictions,
        average="micro",
        zero_division=0
    ),
    "Micro F1": f1_score(
        y_test_hydraulics_final,
        cnn_predictions,
        average="micro",
        zero_division=0
    ),
    "Macro F1": f1_score(
        y_test_hydraulics_final,
        cnn_predictions,
        average="macro",
        zero_division=0
    ),
    "Exact Match": accuracy_score(
        y_test_hydraulics_final,
        cnn_predictions
    )
}


# ============================================================
# STEP 3: Create a comparison table
# ============================================================

hydraulics_model_comparison = pd.DataFrame([
    mlp_metrics_hydraulics,
    cnn_metrics_hydraulics
])

print("=" * 70)
print("HYDRAULICS RCA - MLP vs 1D CNN PERFORMANCE")
print("=" * 70)
display(hydraulics_model_comparison.round(4))


# ============================================================
# STEP 4: Per-root-cause F1 comparison
# ============================================================

mlp_f1_per_cause_hydraulics = f1_score(
    y_test_hydraulics_final,
    mlp_predictions,
    average=None,
    zero_division=0
)

cnn_f1_per_cause_hydraulics = f1_score(
    y_test_hydraulics_final,
    cnn_predictions,
    average=None,
    zero_division=0
)

per_cause_comparison_hydraulics = pd.DataFrame({
    "Root Cause": y_hydraulics.columns,
    "MLP F1": mlp_f1_per_cause_hydraulics,
    "CNN F1": cnn_f1_per_cause_hydraulics
})

print("\n" + "=" * 70)
print("PER ROOT-CAUSE F1 COMPARISON (HYDRAULICS)")
print("=" * 70)
display(per_cause_comparison_hydraulics.round(4))


# ============================================================
# STEP 5: Determine better model and provide guidance
# ============================================================

mlp_f1_value_hydraulics = mlp_metrics_hydraulics["Micro F1"]
cnn_f1_value_hydraulics = cnn_metrics_hydraulics["Micro F1"]

print("\n" + "=" * 70)
print("BEST MODEL FOR HYDRAULICS RCA")
print("=" * 70)

if mlp_f1_value_hydraulics > cnn_f1_value_hydraulics:
    print("MLP has a higher Micro F1-score for Hydraulics RCA.")
elif cnn_f1_value_hydraulics > mlp_f1_value_hydraulics:
    print("1D CNN has a higher Micro F1-score for Hydraulics RCA.")
else:
    print("MLP and 1D CNN have similar Micro F1-scores for Hydraulics RCA.")

print("\nImportant metrics:")
print("Lower Hamming Loss = better")
print("Higher Precision   = better")
print("Higher Recall      = better")
print("Higher F1          = better")
print("Higher Exact Match = better")

# 4. Probe Root Cause Analysis

## 4.1 Probe Feature Extraction, Label Encoding, and Preprocessing

This consolidated pipeline discovers Probe fault files, extracts features around the fault window, creates multi-label root-cause targets, removes highly sparse features, imputes missing values, standardizes features, and prepares train/test datasets.

In [ ]:
# ============================================================
# PROBE RCA - PREPROCESSING
# ============================================================

import os
import glob
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

print("=" * 70)
print("PROBE RCA - PREPROCESSING")
print("=" * 70)

# ------------------------------------------------------------
# 1. FIND PROBE FAULT FILES
# ------------------------------------------------------------

probe_files = glob.glob(
    "causrca_data/dig_twin/exp_probe/**/*.csv",
    recursive=True
)

probe_files = [
    f for f in probe_files
    if "faultDataset" in os.path.basename(f)
]

print(f"\nProbe fault files found: {len(probe_files)}")


# ------------------------------------------------------------
# 2. FEATURE EXTRACTION FUNCTION
# ------------------------------------------------------------

def extract_probe_features(df, cause_start, window=30):

    features = {}

    # Ensure correct time column
    df["time_s"] = pd.to_numeric(df["time_s"], errors="coerce")

    # Remove invalid timestamps
    df = df.dropna(subset=["time_s"])

    before = df[
        (df["time_s"] >= cause_start - window) &
        (df["time_s"] < cause_start)
    ]

    after = df[
        (df["time_s"] >= cause_start) &
        (df["time_s"] <= cause_start + window)
    ]

    # --------------------------------------------------------
    # Process every node
    # --------------------------------------------------------

    for node in df["node"].dropna().unique():

        node_df = df[df["node"] == node]

        node_type_values = node_df["type"].dropna().astype(str)

        if len(node_type_values) == 0:
            continue

        node_type = node_type_values.iloc[0].lower()

        b = before[before["node"] == node].copy()
        a = after[after["node"] == node].copy()

        # ====================================================
        # CONTINUOUS
        # ====================================================

        if "continuous" in node_type:

            b_val = pd.to_numeric(b["value"], errors="coerce").dropna()
            a_val = pd.to_numeric(a["value"], errors="coerce").dropna()

            if len(b_val) > 0:
                b_mean = b_val.mean()
                b_std = b_val.std()

                features[f"{node}_before_mean"] = b_mean
                features[f"{node}_before_std"] = b_std

            else:
                b_mean = np.nan
                b_std = np.nan

                features[f"{node}_before_mean"] = np.nan
                features[f"{node}_before_std"] = np.nan

            if len(a_val) > 0:
                a_mean = a_val.mean()
                a_std = a_val.std()

                features[f"{node}_after_mean"] = a_mean
                features[f"{node}_after_std"] = a_std

            else:
                a_mean = np.nan
                a_std = np.nan

                features[f"{node}_after_mean"] = np.nan
                features[f"{node}_after_std"] = np.nan

            features[f"{node}_mean_change"] = (
                a_mean - b_mean
                if pd.notna(a_mean) and pd.notna(b_mean)
                else np.nan
            )

            features[f"{node}_absolute_change"] = (
                abs(a_mean - b_mean)
                if pd.notna(a_mean) and pd.notna(b_mean)
                else np.nan
            )

            if pd.notna(b_mean) and b_mean != 0 and pd.notna(a_mean):
                features[f"{node}_relative_change"] = (
                    (a_mean - b_mean) / abs(b_mean)
                )
            else:
                features[f"{node}_relative_change"] = np.nan

            features[f"{node}_std_change"] = (
                a_std - b_std
                if pd.notna(a_std) and pd.notna(b_std)
                else np.nan
            )


        # ====================================================
        # BINARY
        # ====================================================

        elif "binary" in node_type:

            b_values = b["value"].astype(str).str.lower()
            a_values = a["value"].astype(str).str.lower()

            true_values = {"true", "1", "yes", "on"}

            b_true = (
                b_values.isin(true_values).mean()
                if len(b_values) > 0 else np.nan
            )

            a_true = (
                a_values.isin(true_values).mean()
                if len(a_values) > 0 else np.nan
            )

            features[f"{node}_before_state"] = b_true
            features[f"{node}_after_state"] = a_true

            features[f"{node}_state_change"] = (
                a_true - b_true
                if pd.notna(a_true) and pd.notna(b_true)
                else np.nan
            )

            # Count state transitions
            combined = pd.concat([
                b_values,
                a_values
            ]).reset_index(drop=True)

            if len(combined) > 1:
                features[f"{node}_transitions"] = (
                    combined != combined.shift()
                ).sum() - 1
            else:
                features[f"{node}_transitions"] = 0


        # ====================================================
        # COUNTER
        # ====================================================

        elif "counter" in node_type:

            b_val = pd.to_numeric(b["value"], errors="coerce").dropna()
            a_val = pd.to_numeric(a["value"], errors="coerce").dropna()

            b_mean = b_val.mean() if len(b_val) > 0 else np.nan
            a_mean = a_val.mean() if len(a_val) > 0 else np.nan

            features[f"{node}_before_mean"] = b_mean
            features[f"{node}_after_mean"] = a_mean

            features[f"{node}_change"] = (
                a_mean - b_mean
                if pd.notna(a_mean) and pd.notna(b_mean)
                else np.nan
            )

            if len(b_val) > 1:
                b_rate = (b_val.iloc[-1] - b_val.iloc[0]) / window
            else:
                b_rate = np.nan

            if len(a_val) > 1:
                a_rate = (a_val.iloc[-1] - a_val.iloc[0]) / window
            else:
                a_rate = np.nan

            features[f"{node}_before_rate"] = b_rate
            features[f"{node}_after_rate"] = a_rate

            features[f"{node}_rate_difference"] = (
                a_rate - b_rate
                if pd.notna(a_rate) and pd.notna(b_rate)
                else np.nan
            )


        # ====================================================
        # CATEGORICAL
        # ====================================================

        elif "categorical" in node_type:

            combined_values = df[df["node"] == node]["value"].dropna()
            categories = combined_values.astype(str).unique()

            for category in categories:

                b_ratio = (
                    (b["value"].astype(str) == str(category)).mean()
                    if len(b) > 0 else np.nan
                )

                a_ratio = (
                    (a["value"].astype(str) == str(category)).mean()
                    if len(a) > 0 else np.nan
                )

                safe_category = str(category).replace(" ", "_")

                features[
                    f"{node}_{safe_category}_before_ratio"
                ] = b_ratio

                features[
                    f"{node}_{safe_category}_after_ratio"
                ] = a_ratio

                features[
                    f"{node}_{safe_category}_change"
                ] = (
                    a_ratio - b_ratio
                    if pd.notna(a_ratio) and pd.notna(b_ratio)
                    else np.nan
                )


        # ====================================================
        # ALARM
        # ====================================================

        elif "alarm" in node_type:

            b_values = b["value"].astype(str).str.lower()
            a_values = a["value"].astype(str).str.lower()

            alarm_values = {
                "true", "1", "yes", "on", "active",
                "alarm", "triggered"
            }

            b_alarm = (
                b_values.isin(alarm_values).mean()
                if len(b_values) > 0 else np.nan
            )

            a_alarm = (
                a_values.isin(alarm_values).mean()
                if len(a_values) > 0 else np.nan
            )

            features[f"{node}_before_alarm_ratio"] = b_alarm
            features[f"{node}_after_alarm_ratio"] = a_alarm

            features[f"{node}_alarm_change"] = (
                a_alarm - b_alarm
                if pd.notna(a_alarm) and pd.notna(b_alarm)
                else np.nan
            )

    return features


# ------------------------------------------------------------
# 3. PROCESS ALL PROBE CASES
# ------------------------------------------------------------

probe_feature_rows = []

for file in probe_files:

    try:

        # Read CSV
        df = pd.read_csv(file)

        required_columns = {"time_s", "node", "value", "type"}

        if not required_columns.issubset(df.columns):
            print("Skipping:", file)
            print("Missing columns:",
                  required_columns - set(df.columns))
            continue

        # Extract experiment and run
        parts = file.replace("\\", "/").split("/")

        exp_index = parts.index("exp_probe")

        experiment = parts[exp_index + 1]
        run = parts[exp_index + 2]

        case_name = f"{experiment}/{run}"

        # ----------------------------------------------------
        # Read causes.json
        # ----------------------------------------------------

        causes_file = os.path.join(
            os.path.dirname(file),
            "causes.json"
        )

        if not os.path.exists(causes_file):
            print("No causes.json:", case_name)
            continue

        with open(causes_file, "r") as f:
            causes = json.load(f)

        # ----------------------------------------------------
        # Get cause start time
        # ----------------------------------------------------

        cause_start = causes.get("cause_start_at")

        if cause_start is None:
            print("No cause_start_at:", case_name)
            continue

        # Convert timestamp if necessary
        if isinstance(cause_start, str):
            try:
                cause_start = float(cause_start)
            except:
                try:
                    cause_start = pd.to_datetime(
                        cause_start
                    ).timestamp()
                except:
                    print("Invalid cause_start:", case_name)
                    continue
        cause_start = float(cause_start)

        # ----------------------------------------------------
        # Extract features
        # ----------------------------------------------------

        features = extract_probe_features(
            df,
            cause_start,
            window=30
        )

        if len(features) == 0:
            print("No features:", case_name)
            continue

        features["case_name"] = case_name
        features["experiment"] = experiment
        features["run"] = run
        features["cause_start_at"] = cause_start

        probe_feature_rows.append(features)

    except Exception as e:

        print(f"Error processing {file}")
        print("Error:", e)


# ------------------------------------------------------------
# 4. CREATE FEATURE MATRIX
# ------------------------------------------------------------

if len(probe_feature_rows) == 0:
    raise ValueError(
        "No Probe cases were successfully processed."
    )

probe_features = pd.DataFrame(probe_feature_rows)

print("\nSuccessfully processed Probe cases:",
      len(probe_features))
print("Original feature matrix:",
      probe_features.shape)


# ------------------------------------------------------------
# 5. REMOVE FEATURES WITH >80% MISSING VALUES
# ------------------------------------------------------------

metadata_columns = [
    "case_name",
    "experiment",
    "run",
    "cause_start_at"
]

feature_columns = [
    c for c in probe_features.columns
    if c not in metadata_columns
]

missing_ratio = (
    probe_features[feature_columns]
    .isna()
    .mean()
)

features_to_keep = missing_ratio[
    missing_ratio <= 0.80
].index.tolist()

probe_features = probe_features[
    metadata_columns + features_to_keep
]

print(
    "After >80% missing-value filtering:",
    probe_features.shape
)

print(
    "Remaining missing values:",
    probe_features[features_to_keep]
    .isna()
    .sum()
    .sum()
)


# ------------------------------------------------------------
# 6. LOAD ROOT-CAUSE LABELS
# ------------------------------------------------------------

description_files = glob.glob(
    "causrca_data/dig_twin/exp_probe/*/*_description.json"
)

print("\nProbe description files found:",
      len(description_files))

experiment_causes = {}

for desc_file in description_files:
    with open(desc_file, "r") as f:
        desc = json.load(f)

    experiment = os.path.basename(
        os.path.dirname(desc_file)
    )

    manipulated_vars = desc.get(
        "manipulatedVars",
        []
    )

    experiment_causes[experiment] = manipulated_vars

print("\nRoot-cause mapping:")
for exp, causes_list in experiment_causes.items():
    print(exp, ":", causes_list)


# ------------------------------------------------------------
# 7. CREATE MULTI-LABEL TARGET
# ------------------------------------------------------------

probe_features["root_causes"] = probe_features[
    "experiment"
].map(experiment_causes)

# Remove cases without labels
probe_features = probe_features[
    probe_features["root_causes"].notna()
].copy()

mlb_probe = MultiLabelBinarizer()

y_probe = mlb_probe.fit_transform(
    probe_features["root_causes"]
)

print("\nRoot-cause classes:")
print(mlb_probe.classes_)

print("\nTarget shape:", y_probe.shape)


# ------------------------------------------------------------
# 8. CREATE X
# ------------------------------------------------------------

X_probe = probe_features[
    features_to_keep
].copy()

print("\nFinal X shape:", X_probe.shape)


# ------------------------------------------------------------
# 9. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train_probe, X_test_probe, \
y_train_probe, y_test_probe = train_test_split(
    X_probe,
    y_probe,
    test_size=0.20,
    random_state=42
)

print("\nTrain shape:", X_train_probe.shape)
print("Test shape:", X_test_probe.shape)


# ------------------------------------------------------------
# 10. IMPUTATION
# ------------------------------------------------------------

imputer_probe = SimpleImputer(
    strategy="median"
)

X_train_probe_imputed = imputer_probe.fit_transform(
    X_train_probe
)

X_test_probe_imputed = imputer_probe.transform(
    X_test_probe
)


# ------------------------------------------------------------
# 11. STANDARDIZATION
# ------------------------------------------------------------

scaler_probe = StandardScaler()

X_train_probe_final = scaler_probe.fit_transform(
    X_train_probe_imputed
)

X_test_probe_final = scaler_probe.transform(
    X_test_probe_imputed
)


# ------------------------------------------------------------
# 12. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PROBE PREPROCESSING COMPLETED")
print("=" * 70)

print("X_train:",
      X_train_probe_final.shape)
print("X_test:",
      X_test_probe_final.shape)
print("y_train:",
      y_train_probe.shape)
print("y_test:",
      y_test_probe.shape)

print("\nNumber of root-cause classes:",
      len(mlb_probe.classes_))

print("\nRoot-cause classes:")
for i, cls in enumerate(mlb_probe.classes_):
    print(i, "->", cls)

print("\nRemaining NaN in X_train:",
      np.isnan(X_train_probe_final).sum())
print("Remaining NaN in X_test:",
      np.isnan(X_test_probe_final).sum())


## 4.2 Probe MLP and 1D CNN Models

The Probe subsystem uses the same complementary model strategy: an MLP for global nonlinear relationships and a 1D CNN for local patterns in the engineered feature representation.

In [ ]:
# ============================================================
# PROBE RCA - MLP AND 1D CNN TRAINING
# ============================================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization,
    Conv1D, MaxPooling1D, Flatten
)
from tensorflow.keras.optimizers import Adam

print("=" * 70)
print("PROBE RCA - MODEL TRAINING")
print("=" * 70)

# ------------------------------------------------------------
# 1. BASIC INFORMATION
# ------------------------------------------------------------

n_features = X_train_probe_final.shape[1]
n_classes = y_train_probe.shape[1]

print("\nNumber of input features :", n_features)
print("Number of root-cause classes :", n_classes)

print("\nRoot-cause classes:")
for i, cls in enumerate(mlb_probe.classes_):
    print(f"{i} -> {cls}")


# ============================================================
# 2. MLP MODEL
# ============================================================

print("\n" + "=" * 70)
print("TRAINING PROBE MLP")
print("=" * 70)

mlp_probe = Sequential([
    Dense(
        128,
        activation="relu",
        input_shape=(n_features,)
    ),
    BatchNormalization(),
    Dropout(0.30),
    Dense(
        64,
        activation="relu"
    ),
    Dropout(0.20),
    Dense(
        32,
        activation="relu"
    ),
    Dense(
        n_classes,
        activation="sigmoid"
    )
])

mlp_probe.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "binary_accuracy"
    ]
)

mlp_probe.summary()

history_mlp_probe = mlp_probe.fit(
    X_train_probe_final,
    y_train_probe,
    epochs=50,
    batch_size=8,
    validation_split=0.20,
    verbose=1
)


# ============================================================
# 3. 1D CNN MODEL
# ============================================================

print("\n" + "=" * 70)
print("TRAINING PROBE 1D CNN")
print("=" * 70)

# CNN requires 3D input:
# samples × features × channels

X_train_probe_cnn = X_train_probe_final.reshape(
    X_train_probe_final.shape[0],
    X_train_probe_final.shape[1],
    1
)

X_test_probe_cnn = X_test_probe_final.reshape(
    X_test_probe_final.shape[0],
    X_test_probe_final.shape[1],
    1
)

print("\nCNN training input shape:",
      X_train_probe_cnn.shape)

cnn_probe = Sequential([
    Conv1D(
        filters=32,
        kernel_size=3,
        activation="relu",
        input_shape=(n_features, 1)
    ),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.25),
    Conv1D(
        filters=64,
        kernel_size=3,
        activation="relu"
    ),
    MaxPooling1D(pool_size=2),
    Dropout(0.25),
    Flatten(),
    Dense(
        64,
        activation="relu"
    ),
    Dropout(0.30),
    Dense(
        n_classes,
        activation="sigmoid"
    )
])

cnn_probe.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[
        "binary_accuracy"
    ]
)

cnn_probe.summary()

history_cnn_probe = cnn_probe.fit(
    X_train_probe_cnn,
    y_train_probe,
    epochs=50,
    batch_size=8,
    validation_split=0.20,
    verbose=1
)


# ============================================================
# 4. FINAL TRAINING SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PROBE MODEL TRAINING COMPLETED")
print("=" * 70)

print("\nMLP parameters:",
      mlp_probe.count_params())

print("CNN parameters:",
      cnn_probe.count_params())

print("\nBoth Probe models have been trained successfully.")

## 4.3 Probe Model Evaluation

The MLP and CNN are evaluated independently using exact-match accuracy, micro/macro precision, recall, F1-score, Hamming loss, and root-cause-wise metrics.

In [ ]:
# ============================================================
# PROBE RCA - MODEL EVALUATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    classification_report
)

print("=" * 70)
print("PROBE RCA - MLP vs 1D CNN EVALUATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. PREDICTIONS
# ------------------------------------------------------------

mlp_prob_probe = mlp_probe.predict(
    X_test_probe_final,
    verbose=0
)

cnn_prob_probe = cnn_probe.predict(
    X_test_probe_cnn,
    verbose=0
)

# Convert probabilities to binary predictions
threshold = 0.5

mlp_pred_probe = (mlp_prob_probe >= threshold).astype(int)
cnn_pred_probe = (cnn_prob_probe >= threshold).astype(int)


# ------------------------------------------------------------
# 2. EVALUATION FUNCTION
# ------------------------------------------------------------

def evaluate_probe_model(name, y_true, y_pred):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    # Exact-match accuracy:
    # Entire multi-label prediction must be correct
    exact_accuracy = accuracy_score(
        y_true,
        y_pred
    )

    # Micro metrics
    micro_precision = precision_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    micro_recall = recall_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    micro_f1 = f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    # Macro metrics
    macro_precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    # Hamming loss
    h_loss = hamming_loss(
        y_true,
        y_pred
    )

    print(f"\nExact-match Accuracy : {exact_accuracy:.4f}")
    print(f"Micro Precision      : {micro_precision:.4f}")
    print(f"Micro Recall         : {micro_recall:.4f}")
    print(f"Micro F1             : {micro_f1:.4f}")

    print(f"\nMacro Precision      : {macro_precision:.4f}")
    print(f"Macro Recall         : {macro_recall:.4f}")
    print(f"Macro F1             : {macro_f1:.4f}")

    print(f"\nHamming Loss         : {h_loss:.4f}")

    # --------------------------------------------------------
    # Root-cause-wise performance
    # --------------------------------------------------------

    print("\nRoot-cause-wise performance:")
    print("-" * 70)

    report = classification_report(
        y_true,
        y_pred,
        target_names=mlb_probe.classes_,
        output_dict=True,
        zero_division=0
    )

    rows = []

    for cls in mlb_probe.classes_:

        rows.append({
            "Root Cause": cls,
            "Precision": report[cls]["precision"],
            "Recall": report[cls]["recall"],
            "F1": report[cls]["f1-score"],
            "Support": report[cls]["support"]
        })

    results = pd.DataFrame(rows)

    display(
        results.style.format({
            "Precision": "{:.3f}",
            "Recall": "{:.3f}",
            "F1": "{:.3f}"
        })
    )

    return {
        "Exact Accuracy": exact_accuracy,
        "Micro Precision": micro_precision,
        "Micro Recall": micro_recall,
        "Micro F1": micro_f1,
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Hamming Loss": h_loss
    }


# ------------------------------------------------------------
# 3. EVALUATE BOTH MODELS
# ------------------------------------------------------------

mlp_probe_results = evaluate_probe_model(
    "PROBE MLP",
    y_test_probe,
    mlp_pred_probe
)

cnn_probe_results = evaluate_probe_model(
    "PROBE 1D CNN",
    y_test_probe,
    cnn_pred_probe
)


# ------------------------------------------------------------
# 4. FINAL COMPARISON
# ------------------------------------------------------------

comparison_probe = pd.DataFrame(
    [
        mlp_probe_results,
        cnn_probe_results
    ],
    index=["MLP", "1D CNN"]
)

print("\n" + "=" * 70)
print("PROBE MODEL COMPARISON")
print("=" * 70)

display(
    comparison_probe.style.format("{:.4f}")
)


# ------------------------------------------------------------
# 5. BEST MODEL
# ------------------------------------------------------------

best_model_probe = comparison_probe[
    "Micro F1"
].idxmax()

print("\nBest Probe model based on Micro F1:",
      best_model_probe)

print("\nProbe evaluation completed.")

# 5. Multi-Model Fusion and Root Cause Analysis

## 5.1 Within-Subsystem Late Fusion

The MLP and CNN probabilities are combined using equal-weight late fusion: **P_fused = 0.5 × P_MLP + 0.5 × P_CNN**. Fusion combines complementary model evidence without retraining a third neural network.

In [ ]:
# ============================================================
# STEP 1: WITHIN-SUBSYSTEM MODEL FUSION
# MLP + CNN
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# We give equal weight to MLP and CNN initially
# ------------------------------------------------------------

MLP_WEIGHT = 0.5
CNN_WEIGHT = 0.5

THRESHOLD = 0.5


def fuse_models(mlp_model, cnn_model, X_test, X_test_cnn):

    mlp_prob = mlp_model.predict(
        X_test,
        verbose=0
    )

    cnn_prob = cnn_model.predict(
        X_test_cnn,
        verbose=0
    )

    fused_prob = (
        MLP_WEIGHT * mlp_prob +
        CNN_WEIGHT * cnn_prob
    )

    fused_pred = (
        fused_prob >= THRESHOLD
    ).astype(int)

    return mlp_prob, cnn_prob, fused_prob, fused_pred


# ============================================================
# COOLANT
# ============================================================

mlp_prob_coolant, \
cnn_prob_coolant, \
fused_prob_coolant, \
fused_pred_coolant = fuse_models(
    mlp_coolant,
    cnn_coolant,
    X_test_coolant_final,
    X_test_coolant_cnn
)


# ============================================================
# HYDRAULICS
# ============================================================

mlp_prob_hydraulics, \
cnn_prob_hydraulics, \
fused_prob_hydraulics, \
fused_pred_hydraulics = fuse_models(
    mlp_hydraulics,
    cnn_hydraulics,
    X_test_hydraulics_final,
    X_test_hydraulics_cnn
)


# ============================================================
# PROBE
# ============================================================

mlp_prob_probe, \
cnn_prob_probe, \
fused_prob_probe, \
fused_pred_probe = fuse_models(
    mlp_probe,
    cnn_probe,
    X_test_probe_final,
    X_test_probe_cnn
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 70)
print("WITHIN-SUBSYSTEM FUSION COMPLETED")
print("=" * 70)

print("\nCoolant fused probability shape:",
      fused_prob_coolant.shape)

print("Hydraulics fused probability shape:",
      fused_prob_hydraulics.shape)

print("Probe fused probability shape:",
      fused_prob_probe.shape)

print("\nMLP weight :", MLP_WEIGHT)
print("CNN weight :", CNN_WEIGHT)
print("Threshold  :", THRESHOLD)

## 5.2 Final Case-Wise Root Cause Analysis

Fused probabilities are ranked for every test case. The highest-ranked root cause and the Top-3 candidate causes are compared with the known multi-label ground truth to produce the final case-wise RCA table and overall Top-1/Top-3 summary.

In [ ]:
# ============================================================
# FINAL RCA EVALUATION
# Top-1, Top-3 and case-wise RCA results
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


# ------------------------------------------------------------
# Function to evaluate one subsystem
# ------------------------------------------------------------

def final_rca_evaluation(
    subsystem,
    fused_prob,
    y_test,
    mlb
):

    # Ensure y_test is a NumPy array for consistent indexing
    if isinstance(y_test, pd.DataFrame):
        y_test_array = y_test.values
    else:
        y_test_array = y_test

    class_names = list(mlb.classes_)

    results = []

    top1_correct = 0
    top3_correct = 0

    for i in range(len(y_test_array)):

        probabilities = fused_prob[i]

        # Sort causes by probability
        ranked_indices = np.argsort(probabilities)[::-1]

        # Top-1
        top1_index = ranked_indices[0]
        top1_cause = class_names[top1_index]

        # Top-3
        top3_indices = ranked_indices[:3]
        top3_causes = [class_names[j] for j in top3_indices]

        # Actual causes
        actual_indices = np.where(y_test_array[i] == 1)[0]
        actual_causes = [class_names[j] for j in actual_indices]

        # Check Top-1
        is_top1_correct = top1_cause in actual_causes

        # Check Top-3
        is_top3_correct = any(
            cause in actual_causes for cause in top3_causes
        )

        if is_top1_correct:
            top1_correct += 1

        if is_top3_correct:
            top3_correct += 1

        results.append({
            "Subsystem": subsystem,
            "Case": i + 1,
            "Actual Root Cause(s)": ", ".join(actual_causes),
            "Top-1 Prediction": top1_cause,
            "Top-1 Probability": round(probabilities[top1_index], 4),
            "Top-3 Predictions": ", ".join(top3_causes),
            "Top-1 Correct": is_top1_correct,
            "Top-3 Correct": is_top3_correct
        })

    # Calculate Top-1 and Top-3 accuracy
    top1_accuracy = top1_correct / len(y_test_array)
    top3_accuracy = top3_correct / len(y_test_array)

    return results, top1_accuracy, top3_accuracy


# ------------------------------------------------------------
# Evaluate Coolant
# ------------------------------------------------------------

coolant_final_results, coolant_top1, coolant_top3 = final_rca_evaluation(
    "Coolant",
    fused_prob_coolant,
    y_test_coolant_final,
    mlb_coolant
)


# ------------------------------------------------------------
# Evaluate Hydraulics
# ------------------------------------------------------------

hydraulics_final_results, hydraulics_top1, hydraulics_top3 = final_rca_evaluation(
    "Hydraulics",
    fused_prob_hydraulics,
    y_test_hydraulics_final,
    mlb_hydraulics
)


# ------------------------------------------------------------
# Evaluate Probe
# ------------------------------------------------------------

probe_final_results, probe_top1, probe_top3 = final_rca_evaluation(
    "Probe",
    fused_prob_probe,
    y_test_probe,
    mlb_probe
)


# ------------------------------------------------------------
# Combine all case-wise results
# ------------------------------------------------------------

final_rca_results = pd.DataFrame(
    coolant_final_results +
    hydraulics_final_results +
    probe_final_results
)


# Display case-wise results
print("=" * 70)
print("FINAL CASE-WISE ROOT CAUSE ANALYSIS")
print("=" * 70)

display(final_rca_results)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

summary = pd.DataFrame({
    "Subsystem": [
        "Coolant",
        "Hydraulics",
        "Probe"
    ],
    "Number of Test Cases": [
        len(y_test_coolant_final),
        len(y_test_hydraulics_final),
        len(y_test_probe)
    ],
    "Top-1 Accuracy": [
        coolant_top1,
        hydraulics_top1,
        probe_top1
    ],
    "Top-3 Accuracy": [
        coolant_top3,
        hydraulics_top3,
        probe_top3
    ]
})


print("\n" + "=" * 70)
print("FINAL RCA PERFORMANCE SUMMARY")
print("=" * 70)

display(summary.round(4))


# ------------------------------------------------------------
# Overall Top-1 / Top-3 accuracy
# ------------------------------------------------------------

total_cases = (
    len(y_test_coolant_final) +
    len(y_test_hydraulics_final) +
    len(y_test_probe)
)

overall_top1 = (
    coolant_top1 * len(y_test_coolant_final) +
    hydraulics_top1 * len(y_test_hydraulics_final) +
    probe_top1 * len(y_test_probe)
) / total_cases

overall_top3 = (
    coolant_top3 * len(y_test_coolant_final) +
    hydraulics_top3 * len(y_test_hydraulics_final) +
    probe_top3 * len(y_test_probe)
) / total_cases


print(f"Overall Top-1 Accuracy : {overall_top1:.4f}")
print(f"Overall Top-3 Accuracy : {overall_top3:.4f}")

## 5.3 Top-3 Candidates from Each Subsystem

The three highest-probability root-cause candidates are retained from each subsystem. This produces a concise candidate set for the final cross-subsystem RCA stage.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# TOP-3 ROOT CAUSES FROM EACH SUBSYSTEM
# ============================================================

def get_top3_predictions(probabilities, mlb, subsystem_name):
    rows = []

    for i in range(len(probabilities)):
        probs = probabilities[i]

        top3_idx = np.argsort(probs)[::-1][:3]

        for rank, idx in enumerate(top3_idx, start=1):
            rows.append({
                "Subsystem": subsystem_name,
                "Case": i + 1,
                "Rank": rank,
                "Root Cause": mlb.classes_[idx],
                "Probability": float(probs[idx])
            })

    return pd.DataFrame(rows)


top3_coolant = get_top3_predictions(
    fused_prob_coolant,
    mlb_coolant,
    "Coolant"
)

top3_hydraulics = get_top3_predictions(
    fused_prob_hydraulics,
    mlb_hydraulics,
    "Hydraulics"
)

top3_probe = get_top3_predictions(
    fused_prob_probe,
    mlb_probe,
    "Probe"
)


top3_all = pd.concat(
    [
        top3_coolant,
        top3_hydraulics,
        top3_probe
    ],
    ignore_index=True
)

print("=" * 70)
print("TOP-3 ROOT CAUSE CANDIDATES FROM EACH SUBSYSTEM")
print("=" * 70)

display(top3_all)

## 5.4 Final Candidate Pool

Candidate causes are aggregated by subsystem and root cause. Their strongest fused probability is used as the evidence score, and the candidates are ranked to form the final RCA pool.

In [ ]:
# ============================================================
# FINAL 9-CANDIDATE RCA POOL
# ============================================================

candidate_pool = (
    top3_all
        .groupby(
            ["Subsystem", "Root Cause"],
            as_index=False
        )
        .agg(
            Best_Probability=("Probability", "max")
        )
)

# Equal weight for all three subsystems
subsystem_weights = {
    "Coolant": 1.0,
    "Hydraulics": 1.0,
    "Probe": 1.0
}

candidate_pool["Subsystem Weight"] = (
    candidate_pool["Subsystem"].map(subsystem_weights)
)

candidate_pool["Final Score"] = (
    candidate_pool["Best_Probability"]
    * candidate_pool["Subsystem Weight"]
)

candidate_pool = candidate_pool.sort_values(
    "Final Score",
    ascending=False
).reset_index(drop=True)

candidate_pool["Final Rank"] = (
    np.arange(len(candidate_pool)) + 1
)

# Maximum of 9 candidates
candidate_pool = candidate_pool.head(9)

print("=" * 70)
print("FINAL RCA CANDIDATE POOL")
print("=" * 70)

display(
    candidate_pool[
        [
            "Final Rank",
            "Subsystem",
            "Root Cause",
            "Best_Probability",
            "Final Score"
        ]
    ]
)

## 5.5 Explainable RCA Summary

A readable explanation is generated for each final candidate using its subsystem, root cause, fused evidence score, and relative contribution to the candidate pool.

In [ ]:
# ============================================================
# EXPLAINABLE FINAL RCA
# ============================================================

xai_final = candidate_pool.copy()

xai_final["Contribution (%)"] = (
    xai_final["Final Score"]
    / xai_final["Final Score"].sum()
) * 100

xai_final["Explanation"] = xai_final.apply(
    lambda row:
        f"{row['Subsystem']} supports "
        f"{row['Root Cause']} with probability "
        f"{row['Best_Probability']:.3f}, giving it a "
        f"final evidence contribution of "
        f"{row['Contribution (%)']:.1f}%",
    axis=1
)

print("=" * 70)
print("EXPLAINABLE FINAL ROOT CAUSE ANALYSIS")
print("=" * 70)

display(
    xai_final[
        [
            "Final Rank",
            "Subsystem",
            "Root Cause",
            "Best_Probability",
            "Final Score",
            "Contribution (%)",
            "Explanation"
        ]
    ]
)


## 5.6 Human-Readable Final RCA Output

The final cell presents the most likely root cause and a ranked list of candidate causes in a concise format suitable for interpretation by engineers or researchers.

In [ ]:
# ============================================================
# FINAL HUMAN-READABLE RCA SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL ROOT CAUSE ANALYSIS SUMMARY")
print("=" * 70)

top = xai_final.iloc[0]

print(f"\nMost likely root cause: {top['Root Cause']}")
print(f"Subsystem: {top['Subsystem']}")
print(f"Evidence score: {top['Final Score']:.4f}")
print(f"Evidence contribution: {top['Contribution (%)']:.1f}%")

print("\nTop 9 RCA candidates:\n")

for _, row in xai_final.iterrows():
    print(
        f"{int(row['Final Rank'])}. "
        f"{row['Root Cause']} "
        f"[{row['Subsystem']}] "
        f"-> Score: {row['Final Score']:.4f}"
    )

print("\nInterpretation:")
print(
    "The final RCA combines the fused MLP and CNN predictions "
    "and retains the Top-3 root-cause candidates from each "
    "manufacturing subsystem. The candidates are then ranked "
    "according to their evidence scores."
)
